# 沐曦GPU运行prov-gigapath说明文档
[prov-gigapath](https://github.com/prov-gigapath/prov-gigapath)  是全切片病理基础模型。


[prov-gigapath](https://github.com/prov-gigapath/prov-gigapath) 由第三方以开源许可证发布，我方未对其源代码作任何修改。您可通过以下链接 https://github.com/prov-gigapath/prov-gigapath 查阅该项目的源代码、许可证全文、版权与归属声明及其他声明文件。

## 1. 安装
### 1.1 环境准备
使用沐曦开发者社区提供的[maca-pytorch镜像](https://developer.metax-tech.com/softnova/docker?chip_name=%E6%9B%A6%E4%BA%91C500%E7%B3%BB%E5%88%97&package_name=maca-pytorch:3.8.0.11-torch2.6-py310-ubuntu24.04-amd64)启动容器：

```bash
docker run -it --name test-gigapath --device=/dev/mxcd --device=/dev/dri --group-add video --shm-size=8G --security-opt seccomp=unconfined --ulimit memlock=-1  --ulimit stack=67108864 -v /workspace:/workspace cr.metax-tech.com/public-library/maca-pytorch:3.8.0.11-torch2.6-py310-ubuntu24.04-amd64 /bin/bash
```
**参数说明：**
- `--device=/dev/mxcd --device=/dev/dri`：挂载沐曦GPU设备
- `--group-add video`：添加video组以访问GPU
- `--shm-size=8G`：设置共享内存大小
- `-v`：挂载工作目录

> 如需使用更新版本，请访问[沐曦开发者|镜像资源中心](https://developer.metax-tech.com/softnova/docker/)获取对应资源。

### 1.2 安装

In [ ]:
%%bash
cd /opt 
git clone https://github.com/prov-gigapath/prov-gigapath
cd prov-gigapath
sed -i "/torch/d" requirements.txt
sed -i "/xformers/d" requirements.txt
pip install -r requirements.txt
pip install -e .

### 1.3 环境验证

In [2]:
!cd /opt/prov-gigapath && git rev-parse HEAD
import torch
assert torch.cuda.is_available()
print('PyTorch version:', torch.__version__)
print('CUDA OK:', torch.cuda.get_device_name(0))

55431e04ff853a08f752a0ba42e6e9c48e60c776
PyTorch version: 2.6.0+metax3.8.0.7
CUDA OK: MetaX C500


## 2. 沐曦GPU 运行效果展示


load GigaPath tile encoder

In [7]:
import timm
from PIL import Image
from torchvision import transforms
import torch
device="cuda:0"
# Older versions of timm have compatibility issues. Please ensure that you use a newer version by running the following command: pip install timm>=1.0.3.
#tile_encoder = timm.create_model("hf_hub:prov-gigapath/prov-gigapath", pretrained=True) 
model_args={
    "img_size": 224,
    "in_chans": 3,
    "patch_size": 16,
    "embed_dim": 1536,
    "depth": 40,
    "num_heads": 24,
    "init_values": 1e-05,
    "mlp_ratio": 5.33334,
    "num_classes": 0
    }
tile_encoder = timm.create_model(model_name='vit_giant_patch14_dinov2',**model_args)
ckpt_path='/tmp/pytorch_model.bin'
state_dict = torch.load(ckpt_path, map_location="cpu")
missing_keys, unexpected_keys = tile_encoder.load_state_dict(state_dict, strict=True)
tile_encoder.to(device)
transform = transforms.Compose(
    [
        transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)

Running inference to extract tile level features:

In [8]:
img_path = "/opt/prov-gigapath/images/prov_normal_000_1.png"
sample_input = transform(Image.open(img_path).convert("RGB")).unsqueeze(0)

tile_encoder
tile_encoder.eval()
with torch.no_grad():
    output = tile_encoder(sample_input.to(device)).squeeze()
    print(f"output.shape={output.shape},output.device={output.device}")

output.shape=torch.Size([1536]),output.device=cuda:0


---
本文档仅提供相关软件的配置与使用说明，不包含亦不分发前述软件的源代码或目标代码，且不涉及对其源代码的任何修改。您按照本文档配置、部署或使用相关软件时，应遵守适用许可证规定的条款及条件。相关软件的源代码、许可证全文、版权与归属声明及其他项目文档，请以其官方网站或原始发布页面为准。 

Copyright (c) 2026 MetaX Integrated Circuits (Shanghai) Co., Ltd. All rights reserved.